In [1]:
import sagemaker
import boto3
import botocore

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name



sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
region_mapping = {
    "af-south-1": "626614931356",
    "il-central-1": "780543022126",
    "ap-east-1": "871362719292",
    "ap-northeast-1": "763104351884",
    "ap-northeast-2": "763104351884",
    "ap-northeast-3": "364406365360",
    "ap-south-1": "763104351884",
    "ap-south-2": "772153158452",
    "ap-southeast-1": "763104351884",
    "ap-southeast-2": "763104351884",
    "ap-southeast-3": "907027046896",
    "ap-southeast-4": "457447274322",
    "ca-central-1": "763104351884",
    "cn-north-1": "727897471807",
    "cn-northwest-1": "727897471807",
    "eu-central-1": "763104351884",
    "eu-central-2": "380420809688",
    "eu-north-1": "763104351884",
    "eu-west-1": "763104351884",
    "eu-west-2": "763104351884",
    "eu-west-3": "763104351884",
    "eu-south-1": "692866216735",
    "eu-south-2": "503227376785",
    "me-south-1": "217643126080",
    "me-central-1": "914824155844",
    "sa-east-1": "763104351884",
    "us-east-1": "763104351884",
    "us-east-2": "763104351884",
    "us-gov-east-1": "446045086412",
    "us-gov-west-1": "442386744353",
    "us-iso-east-1": "886529160074",
    "us-isob-east-1": "094389454867",
    "us-west-1": "763104351884",
    "us-west-2": "763104351884",
}

llm_image = f"{region_mapping[sess.boto_region_name]}.dkr.ecr.{sess.boto_region_name}.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi1.3.3-gpu-py310-cu121-ubuntu20.04-v1.0"

print(f"llm image uri: {llm_image}")

llm image uri: 763104351884.dkr.ecr.eu-central-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi1.3.3-gpu-py310-cu121-ubuntu20.04-v1.0


In [3]:
llm_image

'763104351884.dkr.ecr.eu-central-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi1.3.3-gpu-py310-cu121-ubuntu20.04-v1.0'

In [4]:
s3_uri = "s3://sagemaker-eu-central-1-025066244860/meta-Llama-3-8B-qlora-2026-03-18-23-03--2026-03-18-23-03-28-903/output/model.tar.gz"

In [5]:
import json
from sagemaker.huggingface import HuggingFaceModel


instance_type = "ml.g5.4xlarge"
number_of_gpu = 1
health_check_timeout = 600

config = {
    "HF_MODEL_ID":               "/opt/ml/model",
    "SM_NUM_GPUS":               json.dumps(number_of_gpu),
    "MAX_INPUT_LENGTH":          json.dumps(2048),  # max tokens user can send in. ~1500 words (1 token ≈ 0.75 words) 1536 * 0.75
    "MAX_BATCH_PREFILL_TOKENS":  json.dumps(4096),  # total tokens TGI can read across all concurrent requests before generating
    "MAX_TOTAL_TOKENS":          json.dumps(4096),  # input + output per request. 2048 in + 2048 out = ~1500 words each way
    "MAX_BATCH_TOTAL_TOKENS":    json.dumps(20480), # total tokens (input + output) TGI can handle across ALL concurrent users at once 1.e 4096  * 5 user
}


llm_model = HuggingFaceModel(model_data=s3_uri , role=role, image_uri=llm_image, env=config)

In [7]:
endpoint_name = "llama3endpoint--v1"

llm = llm_model.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=1,
    instance_type=instance_type,
    container_startup_health_check_timeout=health_check_timeout,
)

-------!

In [18]:

import json

system_prompt = """You are a helpful and empathetic customer service assistant.
Do not introduce yourself with any name.
Always be polite, professional, and solution-focused.
Produce one answer only.
Do not continue the conversation.
Do not generate another system, user, or assistant turn.
Do not invent placeholders, names, phone numbers, or links.
Keep the answer short and action-focused."""

prompt = "How do I make an order?"

input_text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>
{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

payload = {
    "inputs": input_text,
    "parameters": {
        "do_sample": False,          # use sampling instead of greedy decoding — more natural, varied responses
        "top_p": 0.7,                # only sample from top 70% probability mass — keeps output focused and relevant
        "temperature": 0.2,          # very low = more deterministic/factual. higher (0.7-1.0) = more creative/random
        "top_k": 40,                 # only consider top 40 token candidates at each step
        "max_new_tokens": 150,      # max tokens the model can generate in its response (~750 words)
        "return_full_text": False,   # return only the generated response, not the original prompt + response
        "repetition_penalty": 1.12,  # slightly penalises repeating the same words/phrases
        "stop": [                    # stop generating when the model outputs the end-of-sequence token
            "<|eot_id|>",
            "<|start_header_id|>system<|end_header_id|>",
            "<|start_header_id|>user<|end_header_id|>",
            "<|start_header_id|>assistant<|end_header_id|>",
        ],
    },
}

response = llm.predict(payload)
print(response[0]["generated_text"])


I'm on it! To place an order, you can follow these simple steps:

1. Visit our website at {{Website URL}} and browse through our products.
2. Once you've found what you're looking for, add the items to your cart by clicking on the "Add to Cart" button next to each item.
3. When you have all the desired items in your cart, proceed to checkout by clicking on the "Checkout" button.
4. You will be prompted to enter your shipping address and contact information. Please provide accurate details so that we can deliver your order smoothly.
5. Review the items in your cart and select the appropriate payment method from the available options.
6. Finally, confirm your order and wait for your package


In [ ]:
llm.delete_endpoint()